In [1]:
device = "cuda:0"

HP_atomsdata = {"pml_rcut": 3.0, "pml_mnn": 12, "iml_rcut": 6.0, "iml_mnn": 24} # HP指hyperparams
HP_feat_dim = {'atom_dim': 64, 'bond_dim': 64, 'ang_dim': 32, 'dih_dim': 16}
HP_nn = {'init': 1, 'pml': 2, 'iml': 4, 'decoder': [64,1], 'pooling': 'sum',
         'num_global_tokens': 2}  # 分子任务推荐: 仅开 global(消融显示分子上 phase 无益、LES 证伪); 详见 docs/长程全局增强方法说明.md
HP_train = {'batch_size': 16, 'max_epochs': 10, 'lr': 1e-2, 'adamw_weight_decay': 1e-3,
            'adamw_betas': (0.9, 0.999), '1cycle_final_div_factor': 1e+4, 'gradient_clip_val': 1.0}


In [2]:
import os
os.environ["DIGNN_ENV"] = device

import sys
sys.path.append('../..')

from DIGNN.data import AtomsData, ase2AtomsData
from DIGNN.utils import AtomIndexMapper
from DIGNN.pl import DataModule, TrainModule_FF, TrainModule
from DIGNN.nn import models as dgm
from DIGNN.visualize import plot_comparison, plot_tsne, plot_umap


import time
import random
import torch
import numpy as np
import pytorch_lightning as pl
from ase.build import molecule

In [3]:
from ase.io import read

qm7 = read('qm7_1000samples.xyz', index=':', format='extxyz')
for i in range(len(qm7)):
    qm7[i].arrays['energy'] = qm7[i].get_potential_energy()
    qm7[i].arrays['force'] = np.zeros_like(qm7[i].positions)
atomsdata = [ase2AtomsData(qm7[i], check_rcut=HP_atomsdata["pml_rcut"], properties=['energy','force']) for i in range(len(qm7))]


## 性质训练测试

In [4]:
data = DataModule(atomsdata, 
                    **HP_atomsdata,
                    test_size=0.2, val_size=0.1,
                    batch_size=HP_train['batch_size'], num_workers=1, store_device='cpu',
                    mapper=AtomIndexMapper(),
                    return_type='cplt',
                    )
data.setup()  # 建议将预处理和训练分开，即提前setup

In [10]:
# 模型定义
model = dgm.DIGNN(encoder=dgm.Encoder(num_species=data.mapper.num_embeddings,
                                            **HP_feat_dim,
                                            pml_rcut=HP_atomsdata["pml_rcut"]+0.2,
                                            bondI_dim=HP_feat_dim['bond_dim'],
                                            iml_rcut=HP_atomsdata["iml_rcut"]+0.2),
                processor=dgm.GCN_Processor(**HP_feat_dim,
                                            pml=HP_nn['pml'],
                                            iml=HP_nn['iml'],
                                            residual=True,
                                            dropout=0.0,
                                            bondI_dim=HP_feat_dim['bond_dim'],
                                            init_nn_layer=HP_nn['init'],
                                            num_global_tokens=HP_nn['num_global_tokens'],
                                            ), 
                decoder=dgm.Decoder(dim=[HP_feat_dim['atom_dim']] + HP_nn['decoder'],
                                    reduce_method=HP_nn['pooling'],
                                    dropout=0.0),
                ).to(device)

In [11]:
train_module = TrainModule(model, 
                           compile_model=True, # linux环境下开启编译，速度更快
                           lr=HP_train['lr'],
                           prop='energy',
                           adamw_weight_decay=HP_train['adamw_weight_decay'],
                           adamw_betas=HP_train['adamw_betas'],
                           onecycle_total_steps=HP_train['max_epochs']*len(data.train_dataloader()), 
                           onecycle_final_div_factor=HP_train['1cycle_final_div_factor'],
                           empty_cache_every_epoch=False,
                           enable_embed_decay=True,
                           )
trainer = pl.Trainer(max_epochs=HP_train['max_epochs'],
                    accelerator="gpu",
                    devices=[int(device.split(":")[-1])], # 单卡训练
                    check_val_every_n_epoch=1,
                    log_every_n_steps=100,
                    precision='16-mixed',
                    gradient_clip_val=HP_train['gradient_clip_val'],
                    benchmark=True,
                    # logger=tb_logger,
                    # callbacks=[checkpoint_callback],
                    )

In [ ]:
trainer.fit(train_module, train_dataloaders=data.train_dataloader(), val_dataloaders=data.val_dataloader())
# 如果预处理和训练放在一起：
# trainer.fit(datamodule=data)

In [8]:
trainer.test(train_module, dataloaders=data.test_dataloader())

In [9]:
preds_ene, targets_ene = train_module.test_results.values()
plot_comparison(target=targets_ene, pred=preds_ene)

In [10]:
features, labels = train_module.extract_features(
    dataloader=data.train_batch+data.val_batch+data.test_batch)

plot_tsne(features, labels, 
          perplexity=10 # 困惑度， 大样本时建议调大
          )


In [ ]:
plot_umap(features, labels, 
          n_neighbors=10,      # 控制局部/全局权衡。小值关注局部，大值关注全局
          min_dist=1          # 控制点之间的最小距离。小值更密，大值更稀疏
          )

## 力场训练测试

力场(FF)训练需对能量求导得到原子受力 `force = -dE/dpos`，`loss.backward()` 再穿过它构成**二阶求导(double backward)**。

**为什么不能直接用 `compile_model=True`？**
`torch.compile(model)` 的 AOTAutograd **不支持编译区内的 double backward**（`RuntimeError: ... does not currently support double backward`）。这是 PyTorch 的长期架构限制，升级版本无法解决。

**解法：`compiled_autograd=True`**
它编译的是 autograd 引擎执行的**反向图**（而非把 model 当黑盒），天然支持二阶链路。实测与 eager 数值完全一致、训练提速约 1.5x。使用要点：
- 内部自动切换为 Lightning **手动优化模式**，把 forward(含 force 求导)与 backward 包在同一 `compiled_autograd` context 内。
- 梯度裁剪需通过 `TrainModule_FF(gradient_clip_val=...)` 传入（手动优化下 `pl.Trainer(gradient_clip_val=...)` 不生效）。
- `compile_dynamic=True`（默认）：DIGNN 每个 batch 形状不同，动态编译可将编译图数降为 1，**大幅减少重编译、缩短启动时间**。首次仍需数十秒编译，属正常现象。

**训练好后的推理用 compile 更快吗？**
实测**不会**。推理只需一阶导（`create_graph=False`，`compile(model)` 可用），但在 DIGNN 这种动态图 + 大量不规则 scatter/gather 的模型上，Inductor 难以有效融合：实测 compile 推理反而略慢于 eager，且首次编译要花数十秒。**推理建议直接用 eager（`calc.Calculator(..., compile_model=False)`）。**

**该不该开启 `compiled_autograd`？—— 编译成本 vs 稳态加速的权衡**
compiled_autograd 的加速来自稳态每步的 ~1.5x，但首次动态编译本身较慢（复杂动态图上可达数分钟）。是否划算取决于训练规模：
- **长训练（推荐开启）**：epoch 多、step 多时，一次性编译成本被大量步数摊薄，~1.5x 的稳态加速显著占优。
- **短训练 / 快速调试（建议关闭）**：只跑几步或频繁改代码时，编译成本无法摊回，`compiled_autograd=False`（纯 eager）反而整体更快。
- 经验判断：预计总步数远大于「编译耗时 ÷ 每步 eager 耗时」时才值得开启。

In [4]:
data = DataModule(atomsdata, 
                    **HP_atomsdata,
                    test_size=0.2, val_size=0.1,
                    batch_size=8, num_workers=1, store_device='cpu',
                    mapper=AtomIndexMapper(),
                    return_type='basic',
                    )
data.setup()  # 建议将预处理和训练分开，即提前setup

Updating topo: 100%|██████████| 26/26 [00:00<00:00, 96.26batch/s]


In [5]:
model = dgm.DIGNN(encoder=dgm.Encoder(num_species=data.mapper.num_embeddings,
                                            **HP_feat_dim,
                                            pml_rcut=HP_atomsdata["pml_rcut"]+0.2,
                                            bondI_dim=HP_feat_dim['bond_dim'],
                                            iml_rcut=HP_atomsdata["iml_rcut"]+0.2),
                processor=dgm.GCN_Processor(**HP_feat_dim,
                                            pml=HP_nn['pml'],
                                            iml=HP_nn['iml'],
                                            residual=True,
                                            dropout=0.0,
                                            bondI_dim=HP_feat_dim['bond_dim'],
                                            init_nn_layer=HP_nn['init'],
                                            num_global_tokens=HP_nn['num_global_tokens'],
                                            ), 
                decoder=dgm.Decoder(dim=[HP_feat_dim['atom_dim']] + HP_nn['decoder'],
                                    reduce_method=HP_nn['pooling'],
                                    dropout=0.0),
                ).to(device)

In [6]:
max_epoch = 5

train_module = TrainModule_FF(model,
                              compiled_autograd=True,   # 力场训练用 compiled_autograd 绕过 double backward 限制
                              compile_dynamic=True,     # 动态形状: 编译图数降为1, 减少重编译(默认)
                              lr=1e-3,
                              energy_weight=0.1,
                              force_weight=1.0,
                              adamw_weight_decay=1e-2,
                              adamw_betas=(0.9, 0.999),
                              onecycle_total_steps=max_epoch*len(data.train_dataloader()),
                              onecycle_final_div_factor=1e+5,
                              gradient_clip_val=1.0,     # 手动优化下须在此传裁剪值(Trainer的gradient_clip_val会失效)
                              )
trainer = pl.Trainer(max_epochs=max_epoch,
                    accelerator="gpu",
                    check_val_every_n_epoch=10,
                    log_every_n_steps=50,
                    benchmark=True,
                    inference_mode=False,   # 力场验证/测试需要梯度求力, 必须关闭 inference_mode
                    )

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [7]:
trainer.fit(train_module, train_dataloaders=data.train_dataloader(), val_dataloaders=data.val_dataloader())
# 如果预处理和训练放在一起：
# trainer.fit(datamodule=data)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA GeForce RTX 3060 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model         │ DIGNN   │  209 K │ train │     0 │
│ 1 │ criterion     │ MSELoss │      0 │ train │     0 │
│ 2 │ mae_criterion │ L1Loss  │      0 │ train │     0 │
└───┴───────────────┴─────────┴────────┴───────┴───────┘

Trainable params: 209 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 209 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 169                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/ushio/miniconda3/envs/tys/lib/python3.13/site-packages/torch/autograd/graph.py:829: UserWarning: Attempting 
to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered 
internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:179.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass

...

/home/ushio/miniconda3/envs/tys/lib/python3.13/site-packages/torch/_inductor/compile_fx.py:282: UserWarning: 
TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting 
`torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(

W0730 21:17:07.839000 1013 site-packages/torch/_inductor/utils.py:1436] [!0/0/0_1] Not enough SMs to use max_autotune_gemm mode
W0730 21:17:57.458000 1013 site-packages/torch/fx/experimental/symbolic_shapes.py:6823] [!1/1/0] _maybe_guard_rel() was called on non-relation expression Eq(s218, 1) | Eq(s371, s218)
W0730 21:17:57.775000 1013 site-packages/torch/fx/experimental/symbolic_shapes.py:6823] [!1/1/0] _maybe_guard_rel() was called on non-relation expression Eq(s108, 1) | Eq(s108, 114)
W0730 21:17:57.952000 1013 site-packages/torch/fx/experimental/symbolic_shapes.py:6823] [!1/1/0] _maybe_guard_rel() was called on non-relation expression Eq(s211, 1) | Eq(s375, s211)
W0730 21:17:58.246000 1013 site-packages/torch/fx/experimental/symbolic_shapes.py:6823] [!1/1/0] _maybe_guard_rel() was called on non-relation expression Eq(s123, 1) | Eq(s123, 193)
W0730 21:17:58.476000 1013 site-packages/torch/fx/experimental/symbolic_shapes.py:6823] [!1/1/0] _maybe_guard_rel() was called on non-relation

W0730 21:19:50.904000 1013 site-packages/torch/fx/experimental/symbolic_shapes.py:6823] [!0/0/1] _maybe_guard_rel() was called on non-relation expression Eq(s1501, 1) | Eq(s1528, s1501)
W0730 21:19:51.417000 1013 site-packages/torch/fx/experimental/symbolic_shapes.py:6823] [!0/0/1] _maybe_guard_rel() was called on non-relation expression Eq(s1432, 1) | Eq(s1630, s1432)
W0730 21:20:16.595000 1013 site-packages/torch/fx/experimental/symbolic_shapes.py:6823] [!0/0/1_1] _maybe_guard_rel() was called on non-relation expression Eq(s1501, 1) | Eq(s1528, s1501)
W0730 21:20:17.095000 1013 site-packages/torch/fx/experimental/symbolic_shapes.py:6823] [!0/0/1_1] _maybe_guard_rel() was called on non-relation expression Eq(s1432, 1) | Eq(s1630, s1432)

Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/ushio/miniconda3/envs/tys/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
trainer.test(train_module, dataloaders=data.test_dataloader())

preds_ene, targets_ene, preds_force, targets_force, atom_num = train_module.test_results.values()
plot_comparison(target=targets_ene, pred=preds_ene, atom_num=atom_num)
# plot_comparison(target=targets_force, pred=preds_force)